# 06 — Robustness audit (perturbation evasion)

Loads cached `robustness_results.json` or demos one perturbation live.
Full CLI: `python scripts/run_robustness.py --quick`

In [ ]:
%matplotlib inline
import json, sys
from pathlib import Path
import pandas as pd

_cwd = Path.cwd()
_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
sys.path.insert(0, str(_root / "src"))

from paths import PROCESSED_DIR
from perturb import PERTURBATIONS, apply_perturbation

RESULTS = PROCESSED_DIR / "robustness_results.json"
print("Available perturbations:", list(PERTURBATIONS))
print("Demo:", apply_perturbation("leetspeak", "you are a disgusting idiot"))

In [ ]:
if RESULTS.exists():
    payload = json.loads(RESULTS.read_text(encoding="utf-8"))
    rows = []
    for model, block in payload.get("models", {}).items():
        row = {"model": model, "baseline_det": block.get("baseline_detection_rate")}
        for pert, stats in block.get("perturbations", {}).items():
            if isinstance(stats, dict):
                row[f"evasion_{pert}"] = stats.get("evasion_rate")
        rows.append(row)
    display(pd.DataFrame(rows).round(3))
else:
    print(f"Missing {RESULTS} — run scripts/run_robustness.py")

fig = _root / "report" / "figures" / "robustness_evasion_heatmap.png"
if fig.exists():
    from IPython.display import Image, display as idisplay
    idisplay(Image(filename=str(fig)))